# Notebook: 03 Training Test
### Purpose: create train and validation datasets, build augmentations, run Trainer, save versioned checkpoints.


In [1]:
import os
import sys

from torch.nn import CrossEntropyLoss


sys.path.append(os.path.abspath(".."))
sys.path.append(os.path.abspath("../src"))

import random

import torch

from src.data.annotations import load_json_annotations
from src.data.augmentations import get_train_augmentations, get_val_augmentations
from src.data.loaders import ImageMaskDataset
from src.models.zoo import MODEL_BUILDERS
from src.training.engine import run_training
from src.utils.config import Config
from src.utils.helpers import init_notebook, p, t, c


config = Config.load()

init_notebook(config.train.seed)

train_dir = config.paths.train_images
annotations_path = config.paths.annotations
entries = load_json_annotations(annotations_path)

# Shuffle entries
random.shuffle(entries)



=== init_notebook ===
Done


#### Dataset split

In [2]:
# Compute number of validation samples (20 percent of dataset)
val_count = max(1, int(0.2 * len(entries)))

# Split validation set, and training set
val_entries = entries[:val_count]
train_entries = entries[val_count:]

# Build augmentation pipelines for training and validation
train_tf = get_train_augmentations(config.train.image_size)
val_tf = get_val_augmentations(config.train.image_size)

# Build dataset objects that load image-mask pairs and apply transforms
train_ds = ImageMaskDataset(train_entries, train_dir, transform = train_tf)
val_ds = ImageMaskDataset(val_entries, train_dir, transform = val_tf)

# Build dataloaders
train_loader = torch.utils.data.DataLoader(
        train_ds,
        batch_size = config.train.batch_size,
        shuffle = True,
        num_workers = config.train.num_workers,
)

val_loader = torch.utils.data.DataLoader(
        val_ds,
        batch_size = config.train.batch_size,
        shuffle = False,
        num_workers = config.train.num_workers,
)

p("Train samples", len(train_ds))
p("Val samples", len(val_ds))

Train samples: 120
Val samples: 30


#### Available Models

In [3]:
p("Models", MODEL_BUILDERS)
#config.show()
p("Batch", config.train.batch_size)
p("Epochs", config.train.epochs)
p("Learning Rate", config.train.learning_rate, precision = 9)
p("Image Size", config.train.image_size)


Models: 12 keys
  simple_cnn: <function create_simple_cnn at 0x000001CC5B8560C0>
  unet: <function create_unet at 0x000001CC6723ED40>
  smp_unet: <function create_smp_unet at 0x000001CC674985E0>
  smp_fpn: <function create_smp_fpn at 0x000001CC67498680>
  smp_linknet: <function create_smp_linknet at 0x000001CC67498720>
  smp_deeplabv3: <function create_smp_deeplabv3 at 0x000001CC674987C0>
  smp_deeplabv3plus: <function create_smp_deeplabv3plus at 0x000001CC67498860>
  segformer: <function create_segformer at 0x000001CC67498900>
  yolov8n: <function create_yolov8n at 0x000001CC67498AE0>
  yolov8s: <function create_yolov8s at 0x000001CC67498B80>
  yolov8m: <function create_yolov8m at 0x000001CC67498C20>
  yolov8l: <function create_yolov8l at 0x000001CC67498CC0>
Batch: 8
Epochs: 2
Learning Rate: 0.010000000
Image Size: 224


#### Run Training

In [4]:
version_root = config.paths.models
trainer = run_training(
        config = config,
        train_loader = train_loader,
        val_loader = val_loader,
        version_root = version_root,
        model_name = "simple_cnn",
)

=== Training started ===
[Info]: GPU: NVIDIA GeForce MX350
[Info]: GPU Memory: 2.15 GB

[Warn]: Epoch 1
[Info]: Train loss 1.0061, Val loss 0.8097, IoU 0.2052 (ind=0.2787,grp=0.1317), Acc 0.7317
[Info]: New best model

[Warn]: Epoch 2
[Info]: Train loss 0.7462, Val loss 0.5127, IoU 0.0958 (ind=0.1905,grp=0.0010), Acc 0.8029
[Info]: New best model
[Warn]: Training complete


In [5]:
from models.zoo import build_model


# Verify tensor types
t("Tensor Type Verification")
sample_img, sample_mask = train_ds[0]
p(f"Image dtype: {sample_img.dtype}, shape: {sample_img.shape}")
p(f"Mask dtype: {sample_mask.dtype}, shape: {sample_mask.shape}", color1 = c.BLUE)
p(f"Mask values: min={sample_mask.min()}, max={sample_mask.max()}", color1 = c.BLACK)
p(f"Mask unique values: {torch.unique(sample_mask)}", color1 = c.BLACK)

# Test a batch
batch_imgs, batch_masks = next(iter(train_loader))
p(f"\nBatch image dtype: {batch_imgs.dtype}, shape: {batch_imgs.shape}")
p(f"Batch mask dtype: {batch_masks.dtype}, shape: {batch_masks.shape}", color1 = c.BLUE)

# Test with model
model = build_model('simple_cnn', in_channels = 3, out_channels = 3)
with torch.no_grad():
    preds = model(batch_imgs[:1])
p(f"\nModel output dtype: {preds.dtype}, shape: {preds.shape}", color1 = c.BLACK)

# Test loss
criterion = CrossEntropyLoss()
loss = criterion(preds, batch_masks[:1])
p(f"Loss computed successfully: {loss.item()}", color1 = c.BLACK)


=== Tensor Type Verification ===
Image dtype: torch.float32, shape: torch.Size([3, 224, 224])
Mask dtype: torch.int64, shape: torch.Size([224, 224])
Mask values: min=0, max=2
Mask unique values: tensor([0, 1, 2])

Batch image dtype: torch.float32, shape: torch.Size([8, 3, 224, 224])
Batch mask dtype: torch.int64, shape: torch.Size([8, 224, 224])

Model output dtype: torch.float32, shape: torch.Size([1, 3, 224, 224])
Loss computed successfully: 1.1704670190811157
